In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'
MINCUBESAMPLES = 100

SRFUNCTIONS = {
    'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
    'sqrt':np.sqrt,'exp':np.exp,'log':np.log,'abs':np.abs,
    'sin':np.sin,'cos':np.cos,'max':np.maximum,'min':np.minimum,
    '_safepow':lambda a,b:np.abs(a)**b}

import re
def _prepare_form(form):
    return re.sub(r'(\w+)\^(\w+)',r'_safepow(\1,\2)',form)

def eval_form(form,columns,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    ns.update(columns)
    ns.update(constants)
    out = eval(_prepare_form(form),ns)
    if np.ndim(out)==0:
        n = len(next(v for v in columns.values() if hasattr(v,'__len__')))
        out = np.full(n,float(out))
    return np.asarray(out,dtype=float)

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS['tp_mean']
STD  = STATS['tp_std']
ZMIN = (0.0 - MEAN) / STD

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds['dsig'].values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lfraw = flat('lf')
    shfraw = flat('shf')
    lhfraw = flat('lhf')
    blraw = flat('bl') if 'bl' in ds else np.zeros(ntime*ds.sizes['lat']*ds.sizes['lon'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsraw = ds['tp'].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)
meankernel = np.mean(kernels,axis=0)
weighted = fields * meankernel[None,:,:] * dsig[None,None,:]
if surfmask is not None:
    weighted = weighted * surfmask[:,None,:]
integrals = weighted.sum(axis=2)
rhraw,thetaeraw,thetaestarraw = integrals[:,0],integrals[:,1],integrals[:,2]

valid = np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw) & np.isfinite(obsraw)
rh,thetae,thetaestar = rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf,shf,lhf = lfraw[valid],shfraw[valid],lhfraw[valid]
bl = blraw[valid]
obs = obsraw[valid]
landmask  = lf > 0.5
oceanmask = lf < 0.5
print(f'Loaded {valid.sum():,} valid samples ({landmask.sum():,} land, {oceanmask.sum():,} ocean)')

In [ ]:
regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in regdf.iterrows()}
SRMODELS = CONFIGS['experiments']['sr']['optimizedeqs']
ORDER = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}
COLORS = {name:SRMODELS[name]['color'] for name in ORDER}
print(f'Loaded {len(ORDER)} optimized equations: {[LABELS[n] for n in ORDER]}')

In [ ]:
def predict_eq(name,columns):
    entry = REGISTRY[name]
    form,constants = entry['form'],entry['constants']
    raw = eval_form(form,columns,constants)
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

def get_columns(**overrides):
    cols = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    cols.update(overrides)
    for eqname,entry in REGISTRY.items():
        if eqname in overrides:
            continue
        cols[eqname] = eval_form(entry['form'],cols,entry['constants'])
    return cols

In [ ]:
BASEVARS = ['rh','thetae','thetaestar','lf','shf','lhf','bl']

def cube_monotonicity_test(name,target_var,expected_sign,other_vars,ncubes,mask=None):
    cols = get_columns()
    targetvals = cols[target_var]
    othervalslist = [cols[v] for v in other_vars]
    nother = len(other_vars)
    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]
    nsatisfied,ntested = 0,0
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]
    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        nsel = sel.sum()
        x = targetvals[sel]
        overrides = {k:cols[k][sel] for k in BASEVARS}
        overrides[target_var] = x
        for v,binarr,edgearr in zip(other_vars,bins,edges):
            ci = cidx
            for j in range(nother-1,-1,-1):
                if other_vars[j] == v:
                    bi = ci % ncubes
                    break
                ci //= ncubes
            midpoint = 0.5 * (edgearr[bi] + edgearr[bi+1])
            overrides[v] = np.full(nsel,midpoint)
        cubecols = get_columns(**overrides)
        p = predict_eq(name,cubecols)
        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,p) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expected_sign >= 0 and slope >= 0) or (expected_sign < 0 and slope <= 0):
            nsatisfied += 1
    return nsatisfied,ntested

def data_cube_monotonicity_test(target_var,expected_sign,other_vars,ncubes,mask=None):
    featurevals = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    targetvals = featurevals[target_var]
    othervalslist = [featurevals[v] for v in other_vars]
    nother = len(other_vars)
    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]
    nsatisfied,ntested = 0,0
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]
    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        x,y = targetvals[sel],obs[sel]
        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,y) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expected_sign >= 0 and slope >= 0) or (expected_sign < 0 and slope <= 0):
            nsatisfied += 1
    return nsatisfied,ntested

In [ ]:
CONSTRAINTS = {
    'PC2':{
        'label':r'$\partial P/\partial \widehat{\mathrm{RH}} \geq 0$',
        'target_var':'rh',
        'expected_sign':1,
        'other_vars':['thetae','thetaestar'],
        'land_only':True},
    'PC3':{
        'label':r'$\partial P/\partial \widehat{\theta_e} \geq 0$',
        'target_var':'thetae',
        'expected_sign':1,
        'other_vars':['rh','thetaestar'],
        'land_only':False},
    'PC4':{
        'label':r'$\partial P/\partial \widehat{\theta_e^*} \leq 0$',
        'target_var':'thetaestar',
        'expected_sign':-1,
        'other_vars':['rh','thetae'],
        'land_only':False}}

NCUBES = [3,4,5,6,7]
print(f'Testing {len(CONSTRAINTS)} constraints across {len(NCUBES)} cube sizes')

In [ ]:
dataresults = {}
for pcname,pc in CONSTRAINTS.items():
    mask = landmask if pc['land_only'] else None
    pcts = []
    for n in NCUBES:
        sat,tot = data_cube_monotonicity_test(pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask=mask)
        pcts.append(sat/max(tot,1)*100)
    dataresults[pcname] = np.mean(pcts)

modelresults = {}
for name in ORDER:
    modelresults[name] = {}
    for pcname,pc in CONSTRAINTS.items():
        mask = landmask if pc['land_only'] else None
        pcts = []
        for n in NCUBES:
            sat,tot = cube_monotonicity_test(name,pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask=mask)
            pcts.append(sat/max(tot,1)*100)
        modelresults[name][pcname] = np.mean(pcts)

In [ ]:
PCLABELS = {k:v['label'] for k,v in CONSTRAINTS.items()}
PCNOTES  = {k:('Land only' if v['land_only'] else 'All') for k,v in CONSTRAINTS.items()}

columns = {}
for pcname in CONSTRAINTS:
    header = f'{PCLABELS[pcname]}'
    columns[header] = [dataresults[pcname]] + [modelresults[name][pcname] for name in ORDER]

index = ['ERA5'] + [LABELS[name] for name in ORDER]
df = pd.DataFrame(columns,index=index)
df.index.name = 'Source'

region_row = pd.DataFrame({col:[PCNOTES[pc] for pc in CONSTRAINTS] for col in ['Region']},
                          index=list(columns.keys())).T

styled = df.style.format('{:.1f}%').set_caption(
    f'Physical constraint satisfaction (%, averaged over N={NCUBES}, {SPLIT} split, '
    f'min {MINCUBESAMPLES} samples/cube). PC2 tested over land only.')
styled

In [ ]:
NCUBE = 5
pc = CONSTRAINTS['PC2']
target_var = pc['target_var']
other_vars = pc['other_vars']

featurevals = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
               'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
targetvals = featurevals[target_var]
othervalslist = [featurevals[v] for v in other_vars]
nother = len(other_vars)

cuberows = []
for region,mask in [('Land',landmask),('Ocean',oceanmask)]:
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),NCUBE+1) for v in othervalslist]
    bins = [np.clip(np.digitize(v,e)-1,0,NCUBE-1) for v,e in zip(othervalslist,edges)]
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * NCUBE + bins[i]
    for cidx in range(NCUBE**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        x,y = targetvals[sel],obs[sel]
        xc = x - x.mean()
        slope = np.dot(xc,y) / (np.dot(xc,xc) + 1e-12)
        cuberows.append({
            'region':region,
            'slope':slope,
            'satisfied':slope >= 0,
            'nsamples':int(sel.sum()),
            'rh_mean':rh[sel].mean(),
            'thetae_mean':thetae[sel].mean(),
            'thetaestar_mean':thetaestar[sel].mean(),
            'precip_mean':obs[sel].mean(),
            'stability':thetaestar[sel].mean() - thetae[sel].mean()})

cubedf = pd.DataFrame(cuberows)
ocean = cubedf[cubedf['region']=='Ocean']
land = cubedf[cubedf['region']=='Land']
print(f'Ocean cubes: {len(ocean)} ({ocean.satisfied.sum()} satisfy PC2, '
      f'{(~ocean.satisfied).sum()} violate)')
print(f'Land cubes:  {len(land)} ({land.satisfied.sum()} satisfy PC2, '
      f'{(~land.satisfied).sum()} violate)')

In [ ]:
fig,axs = pplt.subplots(ncols=3,figwidth=6.5,refheight=2.5,sharey=False)

ax = axs[0]
sat_slopes = ocean[ocean['satisfied']]['slope'].values
vio_slopes = ocean[~ocean['satisfied']]['slope'].values
bins_hist = np.linspace(min(ocean['slope'].min(),-0.5),ocean['slope'].max(),30)
ax.hist(sat_slopes,bins=bins_hist,color='#2355a1',alpha=0.7,label=f'Satisfied (n={len(sat_slopes)})')
ax.hist(vio_slopes,bins=bins_hist,color='#C44E52',alpha=0.7,label=f'Violated (n={len(vio_slopes)})')
ax.axvline(0,color='k',ls='--',lw=0.8)
ax.format(xlabel=r'$\partial P / \partial \widehat{\mathrm{RH}}$ slope',ylabel='Number of cubes',
          title='Slope magnitude')
ax.legend(loc='ur',fontsize=7)

ax = axs[1]
ax.scatter(ocean[ocean['satisfied']]['rh_mean'],ocean[ocean['satisfied']]['stability'],
           c='#2355a1',s=20,alpha=0.6,label='Satisfied',zorder=2)
ax.scatter(ocean[~ocean['satisfied']]['rh_mean'],ocean[~ocean['satisfied']]['stability'],
           c='#C44E52',s=40,alpha=0.8,label='Violated',marker='x',zorder=3)
ax.format(xlabel=r'Cube mean $\widehat{\mathrm{RH}}$',
          ylabel=r'Cube mean $\widehat{\theta_e^*} - \widehat{\theta_e}$ (stability)',
          title='Thermodynamic regime')
ax.legend(loc='ur',fontsize=7)

ax = axs[2]
ax.scatter(ocean[ocean['satisfied']]['rh_mean'],ocean[ocean['satisfied']]['precip_mean'],
           c='#2355a1',s=20,alpha=0.6,label='Satisfied',zorder=2)
ax.scatter(ocean[~ocean['satisfied']]['rh_mean'],ocean[~ocean['satisfied']]['precip_mean'],
           c='#C44E52',s=40,alpha=0.8,label='Violated',marker='x',zorder=3)
ax.format(xlabel=r'Cube mean $\widehat{\mathrm{RH}}$',ylabel='Cube mean precip (mm)',
          title='Precipitation regime')
ax.legend(loc='ur',fontsize=7)

axs.format(abc=True,titleloc='l')
pplt.show()
fig.save('../figs/fig_S2.jpg')